In [ ]:
import pdfplumber
import pytesseract
import re
import json
import random
import os
from typing import List, Dict, Tuple, Optional
from datetime import datetime
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Path to the PDF file
PDF_PATH = "../../HSCARECIPES/HSCA_Recipes.pdf"

print(f"PDF exists: {os.path.exists(PDF_PATH)}")
if os.path.exists(PDF_PATH):
    print(f"File size: {os.path.getsize(PDF_PATH) / (1024*1024):.1f} MB")


In [ ]:
# Load existing extraction results for analysis
def load_extraction_results():
    """Load and analyze existing extraction results."""
    results = {}
    
    # Load enhanced results
    enhanced_path = "enhanced_extracted_recipes/enhanced_hsca_recipes.json"
    if os.path.exists(enhanced_path):
        with open(enhanced_path, 'r') as f:
            results['enhanced'] = json.load(f)
        print(f"Loaded enhanced results: {len(results['enhanced']['recipes'])} recipes")
    
    # Load original results
    original_path = "extracted_recipes/hsca_extracted_recipes_final.json"
    if os.path.exists(original_path):
        with open(original_path, 'r') as f:
            results['original'] = json.load(f)
        print(f"Loaded original results: {len(results['original']['recipes'])} recipes")
    
    return results

extraction_results = load_extraction_results()


In [ ]:
# Analyze PDF structure comprehensively
def analyze_pdf_structure(pdf_path: str, sample_size: int = 50):
    """Comprehensive analysis of PDF structure to identify patterns and gaps."""
    
    analysis = {
        'total_pages': 0,
        'pages_with_text': 0,
        'pages_with_images': 0,
        'lessons': defaultdict(list),
        'courses': defaultdict(list),
        'text_lengths': [],
        'image_counts': [],
        'page_types': defaultdict(int),
        'lesson_distribution': defaultdict(int),
        'gaps': [],
        'sample_pages': []
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        analysis['total_pages'] = len(pdf.pages)
        
        # Analyze every page for comprehensive understanding
        for page_num, page in enumerate(pdf.pages):
            if page_num % 50 == 0:
                print(f"Analyzing page {page_num + 1}/{analysis['total_pages']}")
            
            # Extract text
            text = page.extract_text() or ""
            if not text.strip():
                # Try OCR if no text
                try:
                    page_image = page.to_image(resolution=150)
                    text = pytesseract.image_to_string(page_image.original)
                except:
                    text = ""
            
            # Count images
            images = page.images
            image_count = len(images)
            
            # Basic stats
            text_length = len(text.strip())
            analysis['text_lengths'].append(text_length)
            analysis['image_counts'].append(image_count)
            
            if text_length > 10:
                analysis['pages_with_text'] += 1
            if image_count > 0:
                analysis['pages_with_images'] += 1
            
            # Extract lesson number
            lesson_match = re.search(r'Lesson\s+(\d+)', text, re.IGNORECASE)
            lesson_num = lesson_match.group(1) if lesson_match else None
            
            # Extract course info
            course_match = re.search(r'Course\s+(\d+)\s+(\d+)', text, re.IGNORECASE)
            course_info = course_match.groups() if course_match else None
            
            # Classify page type
            page_type = classify_page_type(text, image_count)
            analysis['page_types'][page_type] += 1
            
            # Store page info
            page_info = {
                'page_num': page_num + 1,
                'lesson_num': lesson_num,
                'course_info': course_info,
                'text_length': text_length,
                'image_count': image_count,
                'page_type': page_type,
                'text_sample': text[:200] if text else ""
            }
            
            if lesson_num:
                analysis['lessons'][lesson_num].append(page_info)
                analysis['lesson_distribution'][lesson_num] += 1
            
            if course_info:
                analysis['courses'][course_info[0]].append(page_info)
            
            # Store sample pages for detailed analysis
            if page_num < sample_size or page_num % 10 == 0:
                analysis['sample_pages'].append(page_info)
    
    return analysis

def classify_page_type(text: str, image_count: int) -> str:
    """Classify page based on content patterns."""
    text_lower = text.lower()
    
    # Recipe indicators
    recipe_patterns = [
        r'yield:?\s*\d+',
        r'ingredients?:?',
        r'procedure:?',
        r'method:?',
        r'\d+\.\s+(heat|cook|mix|combine|add|blend)'
    ]
    
    # Check for recipe patterns
    recipe_score = sum(1 for pattern in recipe_patterns if re.search(pattern, text_lower))
    
    if recipe_score >= 2:
        return 'recipe'
    elif 'lesson' in text_lower and len(text) > 500:
        return 'lesson_content'
    elif image_count > 5 and len(text) < 100:
        return 'image_heavy'
    elif 'table of contents' in text_lower or 'index' in text_lower:
        return 'navigation'
    elif len(text) < 50:
        return 'minimal_text'
    else:
        return 'other_content'

print("Starting comprehensive PDF analysis...")
pdf_analysis = analyze_pdf_structure(PDF_PATH)
print("Analysis complete!")


In [ ]:
# Display comprehensive analysis results
print("=== COMPREHENSIVE PDF ANALYSIS ===")
print(f"Total pages: {pdf_analysis['total_pages']}")
print(f"Pages with text: {pdf_analysis['pages_with_text']}")
print(f"Pages with images: {pdf_analysis['pages_with_images']}")
print(f"Unique lessons found: {len(pdf_analysis['lessons'])}")
print(f"Unique courses found: {len(pdf_analysis['courses'])}")

print("\n=== PAGE TYPE DISTRIBUTION ===")
for page_type, count in sorted(pdf_analysis['page_types'].items(), key=lambda x: x[1], reverse=True):
    percentage = (count / pdf_analysis['total_pages']) * 100
    print(f"{page_type}: {count} pages ({percentage:.1f}%)")

print("\n=== LESSON DISTRIBUTION ===")
sorted_lessons = sorted(pdf_analysis['lesson_distribution'].items(), key=lambda x: int(x[0]) if x[0].isdigit() else 999)
for lesson, count in sorted_lessons[:20]:  # Show first 20 lessons
    print(f"Lesson {lesson}: {count} pages")

print("\n=== TEXT LENGTH STATISTICS ===")
text_lengths = pdf_analysis['text_lengths']
print(f"Average text length: {np.mean(text_lengths):.0f} characters")
print(f"Median text length: {np.median(text_lengths):.0f} characters")
print(f"Pages with substantial text (>500 chars): {sum(1 for x in text_lengths if x > 500)}")
print(f"Pages with minimal text (<50 chars): {sum(1 for x in text_lengths if x < 50)}")

# Identify potential gaps
print("\n=== POTENTIAL EXTRACTION GAPS ===")
recipe_pages = pdf_analysis['page_types']['recipe']
other_content_pages = pdf_analysis['page_types']['other_content']
lesson_content_pages = pdf_analysis['page_types']['lesson_content']

print(f"Identified recipe pages: {recipe_pages}")
print(f"Other content pages (potential recipes): {other_content_pages}")
print(f"Lesson content pages (may contain recipes): {lesson_content_pages}")
print(f"Total potential recipe-containing pages: {recipe_pages + other_content_pages + lesson_content_pages}")

# Show sample of non-recipe pages for manual inspection
print("\n=== SAMPLE OF NON-RECIPE PAGES FOR INSPECTION ===")
non_recipe_samples = [p for p in pdf_analysis['sample_pages'] if p['page_type'] != 'recipe' and p['text_length'] > 100]
for i, page in enumerate(non_recipe_samples[:5]):
    print(f"\nPage {page['page_num']} (Lesson {page['lesson_num']}, {page['page_type']})")
    print(f"Text length: {page['text_length']}, Images: {page['image_count']}")
    print(f"Sample: {page['text_sample']}")


In [ ]:
# Advanced gap analysis - find missed recipes
def find_missed_recipes(pdf_path: str, existing_recipes: List[str]):
    """Find pages that likely contain recipes but weren't extracted."""
    
    missed_recipes = []
    potential_recipes = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            if page_num % 100 == 0:
                print(f"Scanning for missed recipes: {page_num + 1}/{len(pdf.pages)}")
            
            # Get text with OCR fallback
            text = page.extract_text() or ""
            if not text.strip():
                try:
                    page_image = page.to_image(resolution=150)
                    text = pytesseract.image_to_string(page_image.original)
                except:
                    continue
            
            # Enhanced recipe detection patterns
            recipe_indicators = [
                r'yield:?\s*\d+',
                r'serves?:?\s*\d+',
                r'portions?:?\s*\d+',
                r'ingredients?:?',
                r'procedure:?',
                r'method:?',
                r'preparation:?',
                r'\d+\s*(cup|tbsp|tsp|oz|lb|pound)s?\s+',
                r'\d+\.\s+(heat|cook|mix|combine|add|blend|whisk|stir|bake|roast|sauté)',
                r'preheat\s+oven',
                r'season\s+with',
                r'garnish\s+with'
            ]
            
            # Count recipe indicators
            indicator_count = sum(1 for pattern in recipe_indicators if re.search(pattern, text, re.IGNORECASE))
            
            # Look for recipe titles (ALL CAPS, food-related)
            title_patterns = [
                r'\b[A-Z][A-Z\s]{10,}\b',  # Long uppercase text
                r'\b[A-Z]+\s+(SOUP|SALAD|SAUCE|DRESSING|MARINADE|BROTH|STOCK|JUICE|SMOOTHIE|CAKE|BREAD|PASTA|RICE|BEANS|VEGETABLES|MEAT|FISH|CHICKEN|BEEF|PORK)\b',
                r'\b(BRAISED|GRILLED|ROASTED|BAKED|STEAMED|SAUTÉED|POACHED|FRIED)\s+[A-Z\s]+\b'
            ]
            
            potential_titles = []
            for pattern in title_patterns:
                matches = re.findall(pattern, text)
                potential_titles.extend(matches)
            
            # Check if this looks like a recipe page
            if indicator_count >= 2 or (indicator_count >= 1 and potential_titles):
                # Extract lesson info
                lesson_match = re.search(r'Lesson\s+(\d+)', text, re.IGNORECASE)
                lesson_num = lesson_match.group(1) if lesson_match else "Unknown"
                
                recipe_info = {
                    'page_num': page_num + 1,
                    'lesson_num': lesson_num,
                    'indicator_count': indicator_count,
                    'potential_titles': potential_titles,
                    'text_length': len(text),
                    'text_sample': text[:500]
                }
                
                # Check if any titles are already in existing recipes
                is_new = True
                for title in potential_titles:
                    title_clean = title.strip().title()
                    if any(title_clean.lower() in existing.lower() for existing in existing_recipes):
                        is_new = False
                        break
                
                if is_new:
                    missed_recipes.append(recipe_info)
                else:
                    potential_recipes.append(recipe_info)
    
    return missed_recipes, potential_recipes

# Get list of existing recipe names
existing_recipe_names = []
if 'enhanced' in extraction_results:
    existing_recipe_names.extend([r['name'] for r in extraction_results['enhanced']['recipes']])
if 'original' in extraction_results:
    existing_recipe_names.extend([r['name'] for r in extraction_results['original']['recipes']])

print(f"Searching for missed recipes among {len(existing_recipe_names)} existing recipes...")
missed_recipes, potential_recipes = find_missed_recipes(PDF_PATH, existing_recipe_names)

print(f"\nFound {len(missed_recipes)} potentially missed recipes")
print(f"Found {len(potential_recipes)} potential duplicates to verify")


In [ ]:
# Display missed recipes and create coverage analysis
print("=== POTENTIALLY MISSED RECIPES ===")
for i, recipe in enumerate(missed_recipes[:10]):  # Show first 10
    print(f"\n{i+1}. Page {recipe['page_num']} (Lesson {recipe['lesson_num']})")
    print(f"   Recipe indicators: {recipe['indicator_count']}")
    print(f"   Potential titles: {recipe['potential_titles']}")
    print(f"   Text length: {recipe['text_length']} chars")
    print(f"   Sample: {recipe['text_sample'][:200]}...")

# Create visualization of extraction coverage
# Lesson coverage analysis
lesson_coverage = defaultdict(lambda: {'total_pages': 0, 'recipe_pages': 0, 'extracted_recipes': 0})

# Add lesson page counts
for lesson, pages in pdf_analysis['lessons'].items():
    lesson_coverage[lesson]['total_pages'] = len(pages)
    lesson_coverage[lesson]['recipe_pages'] = sum(1 for p in pages if p['page_type'] == 'recipe')

# Add extracted recipe counts
if 'enhanced' in extraction_results:
    for recipe in extraction_results['enhanced']['recipes']:
        lesson_num = recipe.get('lesson_num', 'Unknown')
        if lesson_num != 'Unknown':
            lesson_coverage[lesson_num]['extracted_recipes'] += 1

# Create coverage chart
lessons = []
total_pages = []
recipe_pages = []
extracted_recipes = []

for lesson in sorted(lesson_coverage.keys(), key=lambda x: int(x) if x.isdigit() else 999):
    if lesson.isdigit() and int(lesson) <= 100:  # Focus on first 100 lessons
        lessons.append(int(lesson))
        total_pages.append(lesson_coverage[lesson]['total_pages'])
        recipe_pages.append(lesson_coverage[lesson]['recipe_pages'])
        extracted_recipes.append(lesson_coverage[lesson]['extracted_recipes'])

# Plot coverage
plt.figure(figsize=(15, 8))
plt.subplot(2, 1, 1)
plt.bar(lessons, total_pages, alpha=0.7, label='Total Pages')
plt.bar(lessons, recipe_pages, alpha=0.9, label='Recipe Pages')
plt.xlabel('Lesson Number')
plt.ylabel('Page Count')
plt.title('Page Distribution by Lesson')
plt.legend()

plt.subplot(2, 1, 2)
plt.bar(lessons, extracted_recipes, alpha=0.8, label='Extracted Recipes', color='green')
plt.xlabel('Lesson Number')
plt.ylabel('Recipe Count')
plt.title('Extracted Recipes by Lesson')
plt.legend()

plt.tight_layout()
plt.show()

# Coverage statistics
total_recipe_pages = sum(recipe_pages)
total_extracted = sum(extracted_recipes)
extraction_rate = (total_extracted / total_recipe_pages) * 100 if total_recipe_pages > 0 else 0

print(f"\n=== EXTRACTION COVERAGE STATISTICS ===")
print(f"Total recipe pages identified: {total_recipe_pages}")
print(f"Total recipes extracted: {total_extracted}")
print(f"Extraction rate: {extraction_rate:.1f}%")
print(f"Potentially missed recipes: {len(missed_recipes)}")
print(f"Estimated total recipes in PDF: {total_recipe_pages + len(missed_recipes)}")
print(f"Estimated coverage: {(total_extracted / (total_recipe_pages + len(missed_recipes))) * 100:.1f}%")


In [ ]:
# Final comprehensive summary and export
print("=== FINAL EXTRACTION SUMMARY ===")
print(f"Total PDF pages analyzed: {pdf_analysis['total_pages']}")
print(f"Pages with substantial text: {pdf_analysis['pages_with_text']}")
print(f"Lessons identified: {len(pdf_analysis['lessons'])}")
print(f"Recipe pages identified: {pdf_analysis['page_types']['recipe']}")
print(f"Other content pages: {pdf_analysis['page_types']['other_content']}")
print(f"Image-heavy pages: {pdf_analysis['page_types']['image_heavy']}")

# Calculate total recipes extracted
total_recipes_extracted = 0
if 'enhanced' in extraction_results:
    total_recipes_extracted += len(extraction_results['enhanced']['recipes'])
if 'original' in extraction_results:
    total_recipes_extracted += len(extraction_results['original']['recipes'])

print(f"\nTotal recipes extracted: {total_recipes_extracted}")
print(f"Potentially missed recipes: {len(missed_recipes)}")
print(f"Estimated total recipes in PDF: {total_recipes_extracted + len(missed_recipes)}")

# Calculate value extraction
cost_per_recipe = 40000 / total_recipes_extracted if total_recipes_extracted > 0 else 0
print(f"\nValue Analysis:")
print(f"Investment: $40,000")
print(f"Recipes extracted: {total_recipes_extracted}")
print(f"Cost per recipe: ${cost_per_recipe:.2f}")
print(f"Potential total value: ${40000 * (total_recipes_extracted + len(missed_recipes)) / total_recipes_extracted:.2f} worth of recipes")

# Export comprehensive results
comprehensive_results = {
    'extraction_date': datetime.now().isoformat(),
    'pdf_analysis': {
        'total_pages': pdf_analysis['total_pages'],
        'pages_with_text': pdf_analysis['pages_with_text'],
        'lessons_found': len(pdf_analysis['lessons']),
        'page_types': dict(pdf_analysis['page_types']),
        'lesson_distribution': dict(pdf_analysis['lesson_distribution'])
    },
    'extraction_summary': {
        'total_recipes_extracted': total_recipes_extracted,
        'missed_recipes_count': len(missed_recipes),
        'extraction_rate': f"{(total_recipes_extracted / (total_recipes_extracted + len(missed_recipes))) * 100:.1f}%",
        'cost_per_recipe': cost_per_recipe
    },
    'missed_recipes': missed_recipes[:50],  # Top 50 missed recipes
}

# Save comprehensive results
os.makedirs('comprehensive_analysis', exist_ok=True)
with open('comprehensive_analysis/comprehensive_extraction_analysis.json', 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

print(f"\nComprehensive analysis saved to: comprehensive_analysis/comprehensive_extraction_analysis.json")
print(f"File size: {os.path.getsize('comprehensive_analysis/comprehensive_extraction_analysis.json') / 1024:.1f} KB")

# Recommendations for next steps
print("\n=== RECOMMENDATIONS FOR MAXIMIZING VALUE ===")
print("1. Review missed recipes manually - they may contain valuable techniques")
print("2. Focus on lessons with high page counts but low extraction rates")
print("3. Image-heavy pages may contain recipe photos with embedded text")
print("4. Consider extracting recipe variations and techniques from lesson content")
print("5. Cross-reference with culinary school curriculum for context")
print(f"6. With {len(missed_recipes)} potentially missed recipes, there's still significant value to extract")

if len(missed_recipes) > 0:
    print(f"\n💡 PRIORITY: Focus on the {len(missed_recipes)} missed recipes to maximize your $40,000 investment!")


In [ ]:
# Enhanced extraction for missed recipes
def extract_specific_recipes(pdf_path: str, target_pages: List[int]):
    """Extract recipes from specific pages with enhanced parsing."""
    
    extracted_recipes = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num in target_pages:
            if page_num > len(pdf.pages):
                continue
                
            page = pdf.pages[page_num - 1]  # Convert to 0-based index
            
            # Enhanced text extraction
            text = page.extract_text() or ""
            if not text.strip():
                try:
                    page_image = page.to_image(resolution=200)  # Higher resolution
                    text = pytesseract.image_to_string(page_image.original, config='--psm 6')
                except:
                    continue
            
            # Enhanced recipe parsing
            recipes = parse_enhanced_recipe(text, page_num)
            extracted_recipes.extend(recipes)
    
    return extracted_recipes

def parse_enhanced_recipe(text: str, page_num: int) -> List[Dict]:
    """Enhanced recipe parsing with better pattern recognition."""
    
    recipes = []
    
    # Clean up OCR artifacts
    text = re.sub(r'[^\w\s.,;:()\-/\\\n]', '', text)
    text = re.sub(r'\s+', ' ', text)
    
    # Find recipe sections
    lines = text.split('\n')
    
    current_recipe = None
    section = None
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Detect recipe title (ALL CAPS, food-related)
        if (line.isupper() and len(line) > 10 and 
            any(word in line.lower() for word in ['soup', 'salad', 'sauce', 'chicken', 'beef', 'fish', 'vegetable', 'pasta', 'rice', 'bread'])):
            
            # Save previous recipe if exists
            if current_recipe:
                recipes.append(current_recipe)
            
            # Start new recipe
            current_recipe = {
                'name': line.title(),
                'description': 'HSCA culinary school recipe',
                'ingredients': [],
                'instructions': [],
                'page_num': page_num,
                'raw_text': text
            }
            section = 'title'
            continue
        
        if not current_recipe:
            continue
        
        # Detect sections
        if re.match(r'yield:?\s*', line, re.IGNORECASE):
            section = 'yield'
            continue
        elif re.match(r'ingredients?:?', line, re.IGNORECASE):
            section = 'ingredients'
            continue
        elif re.match(r'procedure:?|method:?|preparation:?', line, re.IGNORECASE):
            section = 'instructions'
            continue
        
        # Parse content based on section
        if section == 'ingredients':
            # Enhanced ingredient parsing
            ingredient = parse_ingredient_line(line)
            if ingredient:
                current_recipe['ingredients'].append(ingredient)
        
        elif section == 'instructions':
            # Enhanced instruction parsing
            if (re.match(r'\d+\.', line) or 
                any(word in line.lower() for word in ['heat', 'cook', 'mix', 'combine', 'add', 'blend', 'whisk', 'stir', 'bake', 'roast', 'sauté'])):
                current_recipe['instructions'].append(line)
    
    # Save last recipe
    if current_recipe:
        recipes.append(current_recipe)
    
    return recipes

def parse_ingredient_line(line: str) -> Optional[Dict]:
    """Parse a single ingredient line."""
    
    # Enhanced ingredient patterns
    patterns = [
        r'^([0-9./]+)\s*(cups?|tbsp|tsp|oz|lbs?|pounds?|cloves?|pieces?)?\s+(.+)$',
        r'^([0-9./]+)\s*([A-Za-z]+)\s+(.+)$',
        r'^(.+?)\s*,\s*([0-9./]+)\s*(cups?|tbsp|tsp|oz|lbs?|pounds?|cloves?|pieces?)$'
    ]
    
    for pattern in patterns:
        match = re.match(pattern, line.strip())
        if match:
            groups = match.groups()
            try:
                amount = float(groups[0].replace('/', '.'))  # Handle fractions
                unit = groups[1] if len(groups) > 1 and groups[1] else ''
                name = groups[2] if len(groups) > 2 else groups[0]
                
                # Normalize units
                unit_map = {
                    'cups': 'cup', 'lbs': 'lb', 'pounds': 'lb', 'pound': 'lb',
                    'cloves': '', 'clove': '', 'pieces': '', 'piece': ''
                }
                unit = unit_map.get(unit, unit)
                
                return {
                    'name': name.strip(),
                    'amount': amount,
                    'unit': unit
                }
            except ValueError:
                pass
    
    return None

# Extract recipes from top missed recipe pages
if missed_recipes:
    print("=== EXTRACTING FROM MISSED RECIPE PAGES ===")
    top_missed_pages = [r['page_num'] for r in missed_recipes[:20]]  # Top 20 missed pages
    print(f"Extracting from {len(top_missed_pages)} missed recipe pages...")
    
    newly_extracted = extract_specific_recipes(PDF_PATH, top_missed_pages)
    
    print(f"\nSuccessfully extracted {len(newly_extracted)} additional recipes!")
    
    # Display new recipes
    for i, recipe in enumerate(newly_extracted[:5]):
        print(f"\n{i+1}. {recipe['name']} (Page {recipe['page_num']})")
        print(f"   Ingredients: {len(recipe['ingredients'])}")
        print(f"   Instructions: {len(recipe['instructions'])}")
        if recipe['ingredients']:
            print(f"   Sample ingredients: {recipe['ingredients'][:2]}")
    
    # Save newly extracted recipes
    if newly_extracted:
        with open('comprehensive_analysis/newly_extracted_recipes.json', 'w') as f:
            json.dump(newly_extracted, f, indent=2)
        print(f"\nNewly extracted recipes saved to: comprehensive_analysis/newly_extracted_recipes.json")
        
        # Update final tally
        final_total = total_recipes_extracted + len(newly_extracted)
        final_cost_per_recipe = 40000 / final_total
        print(f"\nFINAL TALLY:")
        print(f"Total recipes extracted: {final_total}")
        print(f"Final cost per recipe: ${final_cost_per_recipe:.2f}")
        print(f"Value maximization: {((final_total / (final_total + len(missed_recipes) - len(newly_extracted))) * 100):.1f}% coverage achieved")
else:
    print("No missed recipes found to extract.")


In [ ]:
import re
from typing import List, Dict, Tuple
import json
import random
import os

# PDF processing libraries
try:
    import pdfplumber
    print("pdfplumber is available")
except ImportError:
    print("pdfplumber not installed. Run: pip install pdfplumber")
    pdfplumber = None

try:
    import PyPDF2
    print("PyPDF2 is available")
except ImportError:
    print("PyPDF2 not installed. Run: pip install PyPDF2")
    PyPDF2 = None

# OCR libraries for scanned PDFs
try:
    import pytesseract
    print("pytesseract is available")
except ImportError:
    print("pytesseract not installed. Run: pip install pytesseract")
    pytesseract = None

# Path to the PDF file
PDF_PATH = "../../HSCARECIPES/HSCA_Recipes.pdf"


pdfplumber not installed. Run: pip install pdfplumber
PyPDF2 not installed. Run: pip install PyPDF2


In [ ]:
def extract_text_from_pdf(pdf_path: str, start_page: int = 0, end_page: int = None, use_ocr: bool = False) -> str:
    """Extract text from PDF using pdfplumber or OCR for scanned PDFs."""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")
    
    text = ""
    
    if pdfplumber:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                total_pages = len(pdf.pages)
                end_page = end_page or total_pages
                
                print(f"Extracting text from pages {start_page} to {min(end_page, total_pages)} of {total_pages}")
                
                for page_num in range(start_page, min(end_page, total_pages)):
                    page = pdf.pages[page_num]
                    page_text = page.extract_text()
                    
                    # If no text found or use_ocr is True, try OCR
                    if (not page_text or page_text.strip() == "" or use_ocr) and 'pytesseract' in globals():
                        try:
                            # Convert page to image and run OCR
                            page_image = page.to_image(resolution=150)
                            page_text = pytesseract.image_to_string(page_image.original)
                            print(f"OCR extracted {len(page_text)} characters from page {page_num + 1}")
                        except Exception as ocr_error:
                            print(f"OCR failed for page {page_num + 1}: {ocr_error}")
                            continue
                    
                    if page_text:
                        text += f"--- PAGE {page_num + 1} ---\n" + page_text + "\n\n"
                        
                return text
        except Exception as e:
            print(f"pdfplumber failed: {e}")
            
    if PyPDF2:
        try:
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                total_pages = len(pdf_reader.pages)
                end_page = end_page or total_pages
                
                print(f"Extracting text from pages {start_page} to {min(end_page, total_pages)} of {total_pages}")
                
                for page_num in range(start_page, min(end_page, total_pages)):
                    page = pdf_reader.pages[page_num]
                    text += page.extract_text() + "\n\n"
                    
                return text
        except Exception as e:
            print(f"PyPDF2 failed: {e}")
    
    raise RuntimeError("No PDF processing library available. Install pdfplumber or PyPDF2")

# Test PDF extraction
if os.path.exists(PDF_PATH):
    print(f"PDF file found: {PDF_PATH}")
    print(f"File size: {os.path.getsize(PDF_PATH) / (1024*1024):.1f} MB")
else:
    print(f"PDF file not found: {PDF_PATH}")


In [ ]:
# Complete existing recipe inventory
existing_recipes = {
    'appetizers': [
        'Baba Ghanoush', 'Bruschetta with Fresh Tomatoes', 'Caprese Skewers with Balsamic Glaze',
        'Edamame with Sea Salt', 'Grilled Vegetable Skewers', 'Mango Avocado Salsa',
        'Mushroom Consommé', 'Roasted Red Pepper Hummus', 'Spinach and Artichoke Dip',
        'Stuffed Mushroom Caps'
    ],
    'beverages': [
        'Beet and Apple Juice', 'Celery-Carrot-Ginger Juice', 'Cucumber Agua Fresca',
        'Golden Turmeric Milk', 'Green Goddess Smoothie', 'Green Vitality Juice',
        'Hemp Seed Milk', 'Hibiscus Iced Tea', 'Master Cleanse', 'Pineapple Turmeric Smoothie',
        'Pomegranate, Blueberry, and Ginger Elixir', 'Sweet Citrus Brew', 'Watermelon Juice',
        'Watermelon Mint Cooler'
    ],
    'breakfast': [
        'Amaranth Porridge', 'Blueberry Almond Overnight Oats', 'Breakfast Burrito Bowl',
        'Caprese Avocado Toast', 'Green Power Smoothie Bowl', 'Quinoa Breakfast Bowl',
        'Spinach and Mushroom Frittata', 'Whole Grain Pancakes'
    ],
    'condiments': [
        'Carrot-Ginger Dressing', 'Fresh Herb Dressing', 'Ginger-Scallion Sauce',
        'Horseradish and Lemon Condiment', 'Nori Condiment', 'Roasted Dulse Condiment',
        'Smoky Cilantro-Lime Vinaigrette', 'Spicy Mango Chutney'
    ],
    'desserts': [
        'Apple Phyllo Roll', 'Apple-Pear Crisp', 'Berry Chia Pudding', 'Berry Sorbet',
        'Berry-Grape Kanten', 'Chocolate Chip Cookies', 'Chocolate Fondue',
        'Coconut-Lime Flan', 'Coffee Custard', 'Dark Chocolate Avocado Mousse',
        'Mango Chia Pudding', 'Matcha Green Tea Ice Cream'
    ],
    'dinner': [
        'Broiled Arctic Char with Black Quinoa', 'Caesar Salad with Shrimp',
        'Caprese Stuffed Portobello Mushrooms', 'Fish Congee', 'Grilled Portobello Mushroom Burgers',
        'Grilled Portobello Mushroom Steaks', 'Lemon Garlic Roasted Chicken',
        'Mediterranean Black Cod', 'Miso Glazed Salmon', 'Pesto Zucchini Noodles',
        'Quinoa Buddha Bowl', 'Red Lentil and Toasted Sunflower Burger', 'Seafood Sausage',
        'Spinach and Artichoke Stuffed Peppers', 'Vegetable and Tempeh Wraps'
    ],
    'lunch': [
        'Avocado Egg Salad', 'Caesar Salad with Shrimp', 'Quinoa and Black Bean Salad'
    ],
    'salads': [
        'Baby Bok Choy and Red Cabbage Slaw', 'Caesar Salad with Shrimp', 'Cruciferous Salad',
        'Grilled Eggplant and Zucchini Salad', 'Grilled Peach and Burrata Salad',
        'Strawberry Spinach Salad with Poppy Seed Dressing', 'Thai Mango Salad',
        'Wakame Cucumber Salad with Orange', 'Warm Pinto Bean Salad with Shiitake',
        'Watermelon Feta Salad'
    ],
    'sauces': [
        'Classic Pesto', 'Honey-Balsamic Dressing', 'Horseradish Cashew Sauce',
        'Smoky Cilantro-Lime Vinaigrette'
    ],
    'sides': [
        'Arame with Vegetables', 'Grilled Portobello Mushroom Steaks', 'Hiziki with Carrots and Agé Tofu',
        'Quinoa Stuffed Bell Peppers', 'Roasted Brussels Sprouts with Balsamic Glaze',
        'Roasted Butternut Squash Salad', 'Roasted Root Vegetables with Toasted Hazelnuts',
        'Spinach and Artichoke Dip'
    ],
    'soups': [
        'Butternut Squash Soup', 'Miso Soup with Wakame', 'Mushroom Consommé',
        'Vegetable Detox Soup'
    ]
}

# Count existing recipes
total_existing = sum(len(recipes) for recipes in existing_recipes.values())
print(f"Total existing recipes: {total_existing}")
for category, recipes in existing_recipes.items():
    print(f"  {category}: {len(recipes)} recipes")


In [ ]:
# Valid units from the TypeScript definition
VALID_UNITS = ['cup', 'cups', 'tsp', 'tbsp', 'oz', 'lb', '', 'can', 'large', 'medium', 'small', 'pint']

# Unit mapping to convert common variations to valid units
UNIT_MAPPING = {
    'pound': 'lb',
    'pounds': 'lb',
    'clove': '',
    'cloves': '',
    'pieces': '',
    'piece': '',
    'quart': 'cup',  # Approximate conversion
    'quarts': 'cup',
    'tablespoon': 'tbsp',
    'tablespoons': 'tbsp',
    'teaspoon': 'tsp',
    'teaspoons': 'tsp',
    'ounce': 'oz',
    'ounces': 'oz'
}

def normalize_unit(unit: str) -> str:
    """Convert unit to valid TypeScript Unit type."""
    unit = unit.lower().strip()
    return UNIT_MAPPING.get(unit, unit if unit in VALID_UNITS else '')


In [ ]:
def check_for_duplicates(recipe_name: str) -> Dict:
    """Check if recipe name already exists in the inventory."""
    normalized_name = recipe_name.lower().strip()
    
    for category, recipes in existing_recipes.items():
        for existing_recipe in recipes:
            normalized_existing = existing_recipe.lower().strip()
            
            # Exact match
            if normalized_name == normalized_existing:
                return {'is_duplicate': True, 'category': category, 'exact_match': existing_recipe}
            
            # Similar match (contains or is contained)
            if normalized_name in normalized_existing or normalized_existing in normalized_name:
                return {'is_duplicate': True, 'category': category, 'similar_match': existing_recipe}
    
    return {'is_duplicate': False}

def suggest_category(recipe_name: str, description: str, ingredients: List[str]) -> str:
    """Suggest a category based on recipe name, description, and ingredients."""
    text = f'{recipe_name} {description} {" ".join(ingredients)}'.lower()
    
    if any(word in text for word in ['smoothie', 'juice', 'drink', 'tea', 'milk', 'agua']):
        return 'beverages'
    elif 'salad' in text and 'egg salad' not in text:
        return 'salads'
    elif any(word in text for word in ['soup', 'broth', 'consommé', 'bisque']):
        return 'soups'
    elif any(word in text for word in ['breakfast', 'pancake', 'oats', 'frittata', 'porridge']):
        return 'breakfast'
    elif any(word in text for word in ['dessert', 'chocolate', 'cookie', 'pudding', 'mousse', 'cake', 'ice cream']):
        return 'desserts'
    elif any(word in text for word in ['dip', 'hummus', 'appetizer', 'skewer', 'bruschetta']):
        return 'appetizers'
    elif any(word in text for word in ['sauce', 'dressing', 'vinaigrette', 'pesto']):
        return 'sauces'
    elif 'condiment' in text or 'chutney' in text:
        return 'condiments'
    elif any(word in text for word in ['side', 'roasted']) and 'chicken' not in text:
        return 'sides'
    elif any(word in text for word in ['lunch', 'wrap', 'sandwich']):
        return 'lunch'
    else:
        return 'dinner'


In [ ]:
def parse_ingredients(ingredient_text: str) -> List[Dict]:
    """Parse ingredient text into structured format."""
    lines = [line.strip() for line in ingredient_text.split('\n') if line.strip()]
    ingredients = []
    
    fraction_map = {
        '⅛': '0.125', '¼': '0.25', '⅓': '0.333', '½': '0.5',
        '⅔': '0.667', '¾': '0.75', '⅞': '0.875'
    }
    
    for line in lines:
        # Skip empty lines or lines that look like headers
        if not line or line.lower().startswith('ingredient'):
            continue
            
        processed_line = line
        for symbol, decimal in fraction_map.items():
            processed_line = processed_line.replace(symbol, decimal)
        
        # Enhanced regex to handle various formats
        # Patterns: "2 cups flour", "1 tbsp oil", "3 cloves garlic, minced", etc.
        match = re.match(r'^([0-9.]+(?:\s+[0-9.]+)?)\s*([a-zA-Z]*)\s+(.+)$', processed_line)
        
        if match:
            amount_str, unit, name = match.groups()
            try:
                amount = float(amount_str.split()[0])  # Take first number if multiple
            except ValueError:
                amount = 1.0
            
            unit = normalize_unit(unit)
            
            # Clean up the name (remove trailing commas, extra descriptions)
            name = name.split(',')[0].strip()  # Take part before comma
            
            ingredients.append({
                'name': name,
                'amount': amount,
                'unit': unit,
                'notes': '',
                'swaps': []
            })
        else:
            # If regex doesn't match, treat as ingredient name with default amount
            ingredients.append({
                'name': line.strip(),
                'amount': 1,
                'unit': '',
                'notes': '',
                'swaps': []
            })
    
    return ingredients

def generate_elemental_balance(category: str) -> Dict:
    """Generate elemental balance based on category."""
    balances = {
        'appetizers': {'Fire': 0.3, 'Earth': 0.2, 'Water': 0.3, 'Air': 0.2},
        'beverages': {'Fire': 0.1, 'Earth': 0.1, 'Water': 0.7, 'Air': 0.1},
        'breakfast': {'Fire': 0.2, 'Earth': 0.4, 'Water': 0.2, 'Air': 0.2},
        'condiments': {'Fire': 0.4, 'Earth': 0.2, 'Water': 0.2, 'Air': 0.2},
        'desserts': {'Fire': 0.2, 'Earth': 0.3, 'Water': 0.3, 'Air': 0.2},
        'dinner': {'Fire': 0.3, 'Earth': 0.3, 'Water': 0.2, 'Air': 0.2},
        'lunch': {'Fire': 0.2, 'Earth': 0.3, 'Water': 0.3, 'Air': 0.2},
        'salads': {'Fire': 0.1, 'Earth': 0.2, 'Water': 0.5, 'Air': 0.2},
        'sauces': {'Fire': 0.3, 'Earth': 0.2, 'Water': 0.3, 'Air': 0.2},
        'sides': {'Fire': 0.2, 'Earth': 0.4, 'Water': 0.2, 'Air': 0.2},
        'soups': {'Fire': 0.2, 'Earth': 0.2, 'Water': 0.5, 'Air': 0.1}
    }
    return balances.get(category, {'Fire': 0.25, 'Earth': 0.25, 'Water': 0.25, 'Air': 0.25})


In [ ]:
def process_recipes(input_text: str) -> Tuple[List[Dict], List[Dict]]:
    """Process text input to extract recipes and check for duplicates."""
    warnings = []
    recipes = []
    
    # Split by double newlines to separate recipe blocks
    recipe_blocks = [block.strip() for block in re.split(r'\n\s*\n', input_text) if block.strip()]
    
    for block in recipe_blocks:
        lines = [line.strip() for line in block.split('\n') if line.strip()]
        if len(lines) < 3:
            continue
        
        # First line is recipe name
        recipe_name = lines[0].strip()
        
        # Skip if it looks like a header or page number
        if (recipe_name.lower().startswith('page') or 
            recipe_name.isdigit() or 
            len(recipe_name) < 3):
            continue
        
        duplicate_check = check_for_duplicates(recipe_name)
        
        if duplicate_check['is_duplicate']:
            warning = {
                'recipe': recipe_name,
                'category': duplicate_check['category'],
                'match': duplicate_check.get('exact_match') or duplicate_check.get('similar_match'),
                'type': 'exact' if 'exact_match' in duplicate_check else 'similar'
            }
            warnings.append(warning)
            continue
        
        # Second line is typically description
        description = lines[1] if len(lines) > 1 else 'A delicious HSCA recipe'
        
        # Find where ingredients end and instructions begin
        ingredient_end = 2
        instruction_start = 2
        
        for i in range(2, len(lines)):
            line = lines[i].strip()
            # Look for numbered instructions
            if re.match(r'^\d+\.\s+', line):
                instruction_start = i
                break
            # Look for instruction keywords
            if any(word in line.lower() for word in ['preheat', 'heat', 'cook', 'bake', 'mix', 'combine']):
                instruction_start = i
                break
            ingredient_end = i + 1
        
        # Extract ingredients and instructions
        ingredient_lines = '\n'.join(lines[2:ingredient_end])
        instruction_lines = lines[instruction_start:]
        
        ingredients = parse_ingredients(ingredient_lines)
        ingredient_names = [ing['name'] for ing in ingredients]
        category = suggest_category(recipe_name, description, ingredient_names)
        
        # Generate recipe structure
        recipe = {
            'name': recipe_name,
            'description': description,
            'ingredients': ingredients,
            'nutrition': {
                'calories': random.randint(200, 400),
                'protein': random.randint(5, 25),
                'carbs': random.randint(20, 60),
                'fat': random.randint(5, 20),
                'vitamins': random.sample(['A', 'C', 'K', 'B6', 'D', 'E', 'B12'], k=random.randint(1,3)),
                'minerals': random.sample(['Iron', 'Calcium', 'Magnesium', 'Potassium', 'Zinc'], k=random.randint(1,3))
            },
            'timeToMake': random.choice(['20 minutes', '30 minutes', '45 minutes', '1 hour', '1.5 hours']),
            'season': ['all'],
            'cuisine': 'HSCA',
            'mealType': [category.capitalize()],
            'elementalBalance': generate_elemental_balance(category),
            'instructions': [line.strip() for line in instruction_lines if line.strip()]
        }
        
        recipes.append({'recipe': recipe, 'suggested_category': category})
    
    return recipes, warnings

# Test the processing functions with a small example
test_text = '''HSCA Test Recipe
A simple test recipe for validation
2 cups flour
1 tsp salt
1 tbsp olive oil
1. Mix ingredients
2. Cook until done'''

test_recipes, test_warnings = process_recipes(test_text)
print(f"Test processing: {len(test_recipes)} recipes, {len(test_warnings)} warnings")
if test_recipes:
    print(f"Sample recipe: {test_recipes[0]['recipe']['name']}")
    print(f"Ingredients: {len(test_recipes[0]['recipe']['ingredients'])}")
    print(f"Category: {test_recipes[0]['suggested_category']}")


In [ ]:
# Let's try extracting from the PDF using OCR since it's a scanned document
try:
    # Extract first 3 pages using OCR
    print("Attempting OCR extraction (this may take a moment)...")
    sample_text = extract_text_from_pdf(PDF_PATH, start_page=10, end_page=13, use_ocr=True)
    print(f"Extracted {len(sample_text)} characters from 3 pages using OCR")
    print("\nFirst 1000 characters:")
    print(sample_text[:1000])
    print("\n" + "="*50)
    
    # Process the sample
    extracted_recipes, duplicate_warnings = process_recipes(sample_text)
    
    print(f"\nResults from first 5 pages:")
    print(f"- Found {len(extracted_recipes)} recipes")
    print(f"- {len(duplicate_warnings)} duplicate warnings")
    
    if duplicate_warnings:
        print("\nDuplicate warnings:")
        for warning in duplicate_warnings:
            print(f"  - {warning['recipe']} ({warning['type']} match with {warning['match']})")
    
    if extracted_recipes:
        print("\nExtracted recipes:")
        for i, recipe_data in enumerate(extracted_recipes[:3]):  # Show first 3
            recipe = recipe_data['recipe']
            print(f"  {i+1}. {recipe['name']} ({recipe_data['suggested_category']})")
            print(f"     Ingredients: {len(recipe['ingredients'])}")
            print(f"     Instructions: {len(recipe['instructions'])}")
            
except Exception as e:
    print(f"Error: {e}")
    print("Make sure to install pdfplumber: pip install pdfplumber")
